In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
_os.environ['AWS_REGION'] = 'us-west-2'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# E2E ML Pipeline with Model Registry
Build a SageMaker Pipeline that processes data, trains a model, and registers it to the Model Registry

In [2]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData
from sagemaker.core.processing import (
    ScriptProcessor,
)
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output
)
from sagemaker.serve.model_builder import ModelBuilder

from sagemaker.core.workflow.parameters import (
    ParameterInteger,
    ParameterString,
)
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.train.configs import Compute

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


In [3]:
# Create the SageMaker Session

sagemaker_session = Session()
pipeline_session = PipelineSession()
sm_client = sagemaker_session.sagemaker_client
region = sagemaker_session.boto_region_name
prefix = "pipeline-v3"

account_id = sagemaker_session.account_id()

In [4]:
# Define variables and parameters needed for the Pipeline steps

role = get_execution_role()
default_bucket = sagemaker_session.default_bucket()
base_job_prefix = "v3-pipeline-example"
s3_prefix = "v3-test-pipeline"
default_bucket_prefix = sagemaker_session.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    s3_prefix = f"{default_bucket_prefix}/{s3_prefix}"
    base_job_prefix = f"{default_bucket_prefix}/{base_job_prefix}"

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputDataUrl",
    default_value=f"s3://notebook-test-engine-ds-674622101542-usw2/datasets/tabular/uci_abalone/abalone.csv",
)
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
hyperparameter_max_depth = ParameterString(name="MaxDepth", default_value="5")

# Cache Pipeline steps to reduce execution time on subsequent executions
cache_config = CacheConfig(enable_caching=True, expire_after="30d")

In [5]:
!mkdir -p code

In [6]:
%%writefile code/preprocess.py

"""Feature engineers the abalone dataset."""
import argparse
import logging
import os
import pathlib
import requests
import tempfile

import boto3
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler())


# Since we get a headerless CSV file we specify the column names here.
feature_columns_names = [
    "sex",
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
]
label_column = "rings"

feature_columns_dtype = {
    "sex": str,
    "length": np.float64,
    "diameter": np.float64,
    "height": np.float64,
    "whole_weight": np.float64,
    "shucked_weight": np.float64,
    "viscera_weight": np.float64,
    "shell_weight": np.float64,
}
label_column_dtype = {"rings": np.float64}


def merge_two_dicts(x, y):
    """Merges two dicts, returning a new copy."""
    z = x.copy()
    z.update(y)
    return z


if __name__ == "__main__":
    logger.debug("Starting preprocessing.")
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-data", type=str, required=True)
    args = parser.parse_args()

    base_dir = "/opt/ml/processing"
    pathlib.Path(f"{base_dir}/data").mkdir(parents=True, exist_ok=True)
    input_data = args.input_data
    bucket = input_data.split("/")[2]
    key = "/".join(input_data.split("/")[3:])

    logger.info("Downloading data from bucket: %s, key: %s", bucket, key)
    fn = f"{base_dir}/data/abalone-dataset.csv"
    s3 = boto3.resource("s3", region_name='us-west-2')
    s3.Bucket(bucket).download_file(key, fn)

    logger.debug("Reading downloaded data.")
    df = pd.read_csv(
        fn,
        header=None,
        names=feature_columns_names + [label_column],
        dtype=merge_two_dicts(feature_columns_dtype, label_column_dtype),
    )
    os.unlink(fn)

    logger.debug("Defining transformers.")
    numeric_features = list(feature_columns_names)
    numeric_features.remove("sex")
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_features = ["sex"]
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    logger.info("Applying transforms.")
    y = df.pop("rings")
    X_pre = preprocess.fit_transform(df)
    y_pre = y.to_numpy().reshape(len(y), 1)

    X = np.concatenate((y_pre, X_pre), axis=1)

    logger.info("Splitting %d rows of data into train, validation, test datasets.", len(X))
    np.random.shuffle(X)
    train, validation, test = np.split(X, [int(0.7 * len(X)), int(0.85 * len(X))])

    logger.info("Writing out datasets to %s.", base_dir)
    pd.DataFrame(train).to_csv(f"{base_dir}/train/train.csv", header=False, index=False)
    pd.DataFrame(validation).to_csv(
        f"{base_dir}/validation/validation.csv", header=False, index=False
    )
    pd.DataFrame(test).to_csv(f"{base_dir}/test/test.csv", header=False, index=False)

Writing code/preprocess.py


In [7]:
# Using modern ScriptProcessor class instead of SKLearnProcessor
sklearn_pipeline_processor = ScriptProcessor(
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        py_version="py3",
        instance_type="ml.m5.xlarge",
    ),
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name=f"{base_job_prefix}-sklearn-preprocess-job",
    sagemaker_session=pipeline_session,
    role=role,
)

pipe_processor_args = sklearn_pipeline_processor.run(
    inputs=[
        ProcessingInput(
            input_name="input-1",
            s3_input=ProcessingS3Input(
                s3_uri=input_data,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
                s3_data_distribution_type="ShardedByS3Key",
            )
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{sagemaker_session.default_bucket()}/{prefix}/train",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob"
            )
        ),
        ProcessingOutput(
            output_name="validation",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{sagemaker_session.default_bucket()}/{prefix}/validation",
                local_path="/opt/ml/processing/validation",
                s3_upload_mode="EndOfJob"
            )
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{sagemaker_session.default_bucket()}/{prefix}/test",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob"
            )
        ),
    ],
    code="code/preprocess.py",
    arguments=["--input-data", input_data],
)

step_process = ProcessingStep(
    name="PreprocessAbaloneData",
    step_args=pipe_processor_args,
    cache_config=cache_config,
)

[07/21/26 07:43:55] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052427;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052428;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


### Training Step

In [8]:
# Define the output path for the model artifacts from the training job
model_path = f"s3://{default_bucket}/{base_job_prefix}/V3PipelineTrain"

image_uri = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.0-1",
    py_version="py3",
    instance_type="ml.m5.xlarge",
)

# Using modern ModelTrainer class instead of Estimator
model_trainer = ModelTrainer(
    training_image=image_uri,
    compute=Compute(
        instance_type=training_instance_type,
        instance_count=1,
    ),
    base_job_name=f"{base_job_prefix}-xgboost-train",
    sagemaker_session=pipeline_session,
    role=role,
    hyperparameters={
        "objective": "reg:linear",
        "num_round": 50,
        "max_depth": hyperparameter_max_depth,
        "eta": 0.2,
        "gamma": 4,
        "min_child_weight": 6,
        "subsample": 0.7,
        "silent": 0,
    },
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        InputData(
            channel_name="validation",
            data_source=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        ),
    ],
)

train_args = model_trainer.train()

step_train = TrainingStep(
    name="TrainModelTestStep",
    step_args=train_args,
    cache_config=cache_config,
)

[07/21/26 07:43:56] INFO     Cannot simulate policies for                                  ]8;id=16052435;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052436;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=16052442;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052443;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=16052450;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16052451;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=16052457;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16052458;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/v3-pipeline-exam                
                             ple-xgboost-train' kms_key_id=None compression_type='GZIP'                            

                    INFO     Training image URI:                                               ]8;id=16052465;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=16052466;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.                     
                             0-1-cpu-py3                                                                           

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052471;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052472;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

### Create model step

In [9]:
# Create Model using ModelBuilder
model_builder = ModelBuilder(
    s3_model_data_url=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    image_uri=image_uri,
    sagemaker_session=pipeline_session,
    role_arn=role,
)

step_create_model = ModelStep(
    name="CreateModel",
    step_args=model_builder.build()
)

[07/21/26 07:43:57] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=16052479;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16052480;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=16052486;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16052487;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052492;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052493;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 07:43:58] DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=16052499;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16052500;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#1388\1388]8;;\
                             MLflow model input                                                                    

                    INFO     Cannot simulate policies for                                  ]8;id=16052505;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052506;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=16052511;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052512;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

                    INFO     Cannot simulate policies for                                  ]8;id=16052517;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052518;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

[07/21/26 07:43:59] WARNING  Could not verify permissions for role                         ]8;id=16052523;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052524;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

sagemaker.config INFO - Applied value(s) from config key = SageMaker.Model.Tags


[07/21/26 07:44:00] INFO     ✅ Model has been created: 'model-1c1d8d52' using server None in ]8;id=16052531;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=16052532;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#3656\3656]8;;\
                             SAGEMAKER_ENDPOINT mode                                                               

### Register Step

In [10]:
step_register_model = ModelStep(
    name="RegisterModel",
    step_args=model_builder.register(
        model_package_group_name="my-model-package-group",
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.xlarge"],
        approval_status="Approved"
    )
)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052537;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052538;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

In [11]:
pipeline = Pipeline(
    name="pipeline-v3",
    parameters=[
        processing_instance_count,
        training_instance_type,
        input_data,
        model_approval_status,
        hyperparameter_max_depth
    ],
    steps=[step_process, step_train, step_create_model, step_register_model],
    sagemaker_session=pipeline_session,
)

In [12]:
pipeline.definition()

[07/21/26 07:44:02] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=16052545;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052546;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=16052551;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052552;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=16052557;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052558;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=16052565;file:///usr/local/lib/python3.12/dist-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=16052566;file:///usr/local/lib/python3.12/dist-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=16052571;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052572;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

'{"Version": "2020-12-01", "Metadata": {}, "Parameters": [{"Name": "ProcessingInstanceCount", "Type": "Integer", "DefaultValue": 1}, {"Name": "TrainingInstanceType", "Type": "String", "DefaultValue": "ml.m5.xlarge"}, {"Name": "InputDataUrl", "Type": "String", "DefaultValue": "s3://notebook-test-engine-ds-674622101542-usw2/datasets/tabular/uci_abalone/abalone.csv"}, {"Name": "ModelApprovalStatus", "Type": "String", "DefaultValue": "PendingManualApproval"}, {"Name": "MaxDepth", "Type": "String", "DefaultValue": "5"}], "PipelineExperimentConfig": {"ExperimentName": {"Get": "Execution.PipelineName"}, "TrialName": {"Get": "Execution.PipelineExecutionId"}}, "MlflowConfig": null, "Steps": [{"Name": "PreprocessAbaloneData", "Type": "Processing", "Arguments": {"ProcessingResources": {"ClusterConfig": {"InstanceType": "ml.m5.xlarge", "InstanceCount": {"Get": "Parameters.ProcessingInstanceCount"}, "VolumeSizeInGB": 30}}, "AppSpecification": {"ImageUri": "246618743249.dkr.ecr.us-west-2.amazonaws.c

In [13]:
pipeline.upsert(role_arn=role)

[07/21/26 07:44:03] INFO     Role                                                          ]8;id=16052578;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052579;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' validated for pipeline. Using it.                                            

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052584;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052585;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Role                                                          ]8;id=16052590;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16052591;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' validated for pipeline. Using it.                                            

[07/21/26 07:44:04] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=16052596;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052597;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[07/21/26 07:44:05] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=16052602;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052603;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

...

[07/21/26 07:44:07] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=16052626;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052627;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[07/21/26 07:44:08] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=16052632;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052633;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=16052638;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052639;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=16052644;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=16052645;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

{'PipelineArn': 'arn:aws:sagemaker:us-west-2:674622101542:pipeline/pipeline-v3',
 'PipelineVersionId': 17,
 'ResponseMetadata': {'RequestId': 'e98e3c9b-2b17-47b9-8279-fde072a64127',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e98e3c9b-2b17-47b9-8279-fde072a64127',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '102',
   'date': 'Tue, 21 Jul 2026 07:44:08 GMT'},
  'RetryAttempts': 0}}

In [14]:
execution = pipeline.start()

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16052650;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16052651;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

In [15]:
execution.wait()